<p align="center">
  <img src="../images/wilson-moses-banner.png" width="100%" alt="Wilson Moses - Data Science and AI Engineering">
</p>


# Loan Approval Data Preparation

**Wilson Moses | AnalystLab Africa Data Science Internship - Week 2**

This notebook documents inspection, cleaning, feature engineering, encoding, scaling, outlier assessment and exploratory feature screening for the public loan-approval practice dataset.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'loan_approval_train.csv'
PROCESSED_DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid')

## Load Dataset

In [ ]:
data = pd.read_csv(RAW_DATA_PATH)
data.head(10)

## What Information is in this data? - Data Types

In [ ]:
data.info()

## What is the shape of the data? - Rows & Columns

In [ ]:
data.shape

## Are there any duplicate records?

In [ ]:
data.duplicated().sum()


**_No duplicate records were identified; therefore, no duplicate observations were removed._**

## Statistical Description - Summary Statistic

In [ ]:
data.describe()


## Data Inspections

### Missing Values

In [ ]:
data.isnull().sum()

### Investigate Categorical Missing Values

#### Gender

In [ ]:
data['Gender'].value_counts(dropna = False)


#### Married

In [ ]:
data['Married'].value_counts(dropna=False)

#### Dependents

In [ ]:
data['Dependents'].value_counts(dropna=False)


### Self Employed

In [ ]:
data['Self_Employed'].value_counts(dropna=False)


### Investigate Numerical Missing Values

#### Loan Amount & Loan_Amount_Term

In [ ]:
data[['LoanAmount', 'Loan_Amount_Term']].describe()

In [ ]:
data['LoanAmount'].median()

In [ ]:
data['LoanAmount'].mean()

### Credit History

In [ ]:
data['Credit_History'].value_counts(dropna=False)

### Numerical Distributions

In [ ]:
data[['ApplicantIncome',
      'CoapplicantIncome',
      'LoanAmount',
      'Loan_Amount_Term']].describe()

## Data Cleaning Preparation

### Gender - Mode Imputation

In [ ]:
data['Gender'] = data['Gender'].fillna(data['Gender'].mode()[0])

Missing Gender values were imputed using the mode because Gender is a categorical variable and only a small proportion of observations were missing. Mode imputation allows the affected observations to be retained without introducing a new category.

### Married - Mode Imputation

In [ ]:
data['Married'] = data['Married'].fillna(data['Married'].mode()[0])

Again, because it is categorical and the number of missing observations is extremely small.

### Dependents - Mode Imputation

In [ ]:
data['Dependents'] = data['Dependents'].fillna(data['Dependents'].mode()[0])

### Self Employed - Mode Imputation

In [ ]:
data['Self_Employed'] = data['Self_Employed'].fillna(data['Self_Employed'].mode()[0])

Missing values represented approximately 5.2% of observations. Since Self_Employed is categorical and the majority of observed applicants were not self-employed, mode imputation was selected to retain observations while avoiding unnecessary row deletion.

### Loan Amount - Median Imputation

In [ ]:
data['LoanAmount'] = data['LoanAmount'].fillna(
    data['LoanAmount'].median()
)

In [ ]:
sns.histplot(data['LoanAmount'], kde=True)
plt.title('Distribution of Loan Amount')
plt.show()

"The distribution contains extreme observations, so median imputation is a more robust choice for missing LoanAmount values."

### Loan_Amount_Term - Mode Imputation

In [ ]:
data['Loan_Amount_Term'] = data['Loan_Amount_Term'].fillna(
    data['Loan_Amount_Term'].mode()[0]
)

In [ ]:
data['Loan_Amount_Term'].value_counts(dropna=False)

In [ ]:
sns.histplot(data['Loan_Amount_Term'], kde=True)
plt.title('Distribution of Loan Amount Term')
plt.show()

Loan_Amount_Term represents a discrete loan term, and the most common actual term is more meaningful than calculating an average term.


Credit_History

In [ ]:
data['Credit_History'].fillna(
    data['Credit_History'].mode()[0]
).value_counts(dropna=False)

In [ ]:
data.groupby('Credit_History')['Loan_Status'].value_counts(normalize=True)

## Cleaning Implementation

In [ ]:
# Categorical variables
categorical_mode_columns = [
    'Gender',
    'Married',
    'Dependents',
    'Self_Employed'
]

for column in categorical_mode_columns:
    data[column] = data[column].fillna(data[column].mode()[0])

# Numerical/discrete variables
data['LoanAmount'] = data['LoanAmount'].fillna(
    data['LoanAmount'].median()
)

data['Loan_Amount_Term'] = data['Loan_Amount_Term'].fillna(
    data['Loan_Amount_Term'].mode()[0]
)

# Binary categorical variable
data['Credit_History'] = data['Credit_History'].fillna(
    data['Credit_History'].mode()[0]
)

### Verify the Cleaning

In [ ]:
data.isnull().sum()

In [ ]:
data.duplicated().sum()

In [ ]:
data.shape

# Feature Engineering

1. Can I represent the existing information better?
2. Can I create a variable that captures useful business meaning?
3. Are there variables that shouldn't go into the model?

## Remove Loan_ID

In [ ]:
data = data.drop(columns=['Loan_ID'])

## Create TotalIncome - ApplicantIncome + CoapplicantIncome

In [ ]:
data['TotalIncome'] = (
    data['ApplicantIncome'] +
    data['CoapplicantIncome']
)

## Create LoanIncomeRatio

### First check whether TotalIncome contains zero

In [ ]:
(data['TotalIncome'] == 0).sum()

In [ ]:
data['LoanIncomeRatio'] = (
    data['LoanAmount'] / data['TotalIncome']
)

## Rename Columns

In [ ]:
data = data.rename(columns={
    'Gender': 'gender',
    'Married': 'married',
    'Dependents': 'dependents',
    'Education': 'education',
    'Self_Employed': 'self_employed',
    'ApplicantIncome': 'applicant_income',
    'CoapplicantIncome': 'coapplicant_income',
    'LoanAmount': 'loan_amount',
    'Loan_Amount_Term': 'loan_amount_term',
    'Credit_History': 'credit_history',
    'Property_Area': 'property_area',
    'Loan_Status': 'loan_status',
    'TotalIncome': 'total_income',
    'LoanIncomeRatio': 'loan_income_ratio'
})

## Checking Resulting Dataset

In [ ]:
data.columns

## Check the Engineered features

In [ ]:
data[['applicant_income',
      'coapplicant_income',
      'total_income',
      'loan_amount',
      'loan_income_ratio']].head(10)

## Check Engineered feature Statistics

In [ ]:
data[['total_income', 'loan_income_ratio']].describe()

## Produce Cleaned Dataset

In [ ]:
cleaned_data = data.copy()

In [ ]:
cleaned_data.isnull().sum()

In [ ]:
cleaned_output_path = PROCESSED_DATA_DIR / 'loan_prediction_cleaned.csv'
cleaned_data.to_csv(cleaned_output_path, index=False)

In [ ]:
cleaned_output_path.exists()

# Feature Encoding

## Binary Categorical Variables

### Gender - Binary encoding

In [ ]:
data['gender'] = data['gender'].map({
    'Male': 1,
    'Female': 0
})

### Married - Binary encoding

In [ ]:
data['married'] = data['married'].map({
    'Yes': 1,
    'No': 0
})

### Education - Binary encoding

In [ ]:
data['education'] = data['education'].map({
    'Graduate': 1,
    'Not Graduate': 0
})

### Self Employed - Binary encoding

In [ ]:
data['self_employed'] = data['self_employed'].map({
    'Yes': 1,
    'No': 0
})

### Credit History

Convert it to int from float - Data type correction

In [ ]:
data['credit_history'] = data['credit_history'].astype(int)

### Dependents - Custom numerical encoding

In [ ]:
data['dependents'] = data['dependents'].replace({
    '3+': '3'
}).astype(int)

Important note:

3 doesn't mean exactly three dependents. It effectively means: three or more dependents

### Property Area - dummy-variable encoding.

In [ ]:
data = pd.get_dummies(
    data,
    columns=['property_area'],
    drop_first=True,
    dtype=int
)

This will produce two columns rather than three. with rural acting as the refernce category

### Encoding the target: Loan Status - Binary Encoding

In [ ]:
data['loan_status'] = data['loan_status'].map({
    'Y': 1,
    'N': 0
})

### Verify the results

In [ ]:
data.head()

In [ ]:
data.info()

In [ ]:
data.dtypes

In [ ]:
data['dependents'].value_counts()

In [ ]:
data.columns

# Feature Scaling

## Inspection of numerical variables

In [ ]:
numerical_features = [
    'applicant_income',
    'coapplicant_income',
    'loan_amount',
    'loan_amount_term',
    'total_income',
    'loan_income_ratio'
]

data[numerical_features].describe()

## Apply StandardScaler

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

data[numerical_features] = scaler.fit_transform(
    data[numerical_features]
)

### Verify Scaling

In [ ]:
data[numerical_features].describe()

## Outlier detection

### Box Plot of Numerical Features

In [ ]:
data[numerical_features].boxplot(figsize=(12, 6))
plt.title('Scaled Numerical Features')
plt.xticks(rotation=45)
plt.show()

### IQR Method

In [ ]:
numerical_features = [
    'applicant_income',
    'coapplicant_income',
    'loan_amount',
    'loan_amount_term',
    'total_income',
    'loan_income_ratio'
]

for column in numerical_features:
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)

    IQR = Q3 - Q1

    if IQR == 0:
        print(f'{column}')
        print('IQR = 0; IQR-based outlier detection is not informative.')
        print('-' * 40)
        continue

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = data[
        (data[column] < lower_bound) |
        (data[column] > upper_bound)
    ]

    print(f'{column}')
    print(f'IQR: {IQR:.3f}')
    print(f'Lower bound: {lower_bound:.3f}')
    print(f'Upper bound: {upper_bound:.3f}')
    print(f'Number of outliers: {len(outliers)}')
    print('-' * 40)

# Feature Selection
1. Which Variables are most relevant?
2. Which variables are redundant?
3. Which features should ultimately be removed?

## We will use three forms of evidence:
1. Correlation
2. Feature Importance
3. Business/feature Logic

### Step 1: Seperate X & Y

In [ ]:
x = data.drop(columns=['loan_status'])
y = data['loan_status']

In [ ]:
x.shape

In [ ]:
y.shape

### Step 2: Correlation Heatmap

In [ ]:
correlation_matrix = data.corr()

In [ ]:
plt.figure(figsize=(14, 10))

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0
)

plt.title('Correlation Heatmap of Loan Prediction Features', fontsize=15, fontweight='bold')
plt.show()


## Investigate redundancy


In [ ]:
data[
    [
        'applicant_income',
        'coapplicant_income',
        'total_income',
        'loan_amount',
        'loan_income_ratio'
    ]
].corr()

## Identify High Correlated pairs

In [ ]:
corr_matrix = x.corr().abs()

upper_triangle = corr_matrix.where(
    np.triu(
        np.ones(corr_matrix.shape),
        k=1
    ).astype(bool)
)

high_corr_pairs = (
    upper_triangle
    .stack()
    .sort_values(ascending=False)
)

high_corr_pairs[high_corr_pairs >= 0.80]

## Step 3 - Examine feature relationships with loan_status

In [ ]:
target_correlation = (
    data.corr()['loan_status']
    .drop('loan_status')
    .sort_values(key=abs, ascending=False)
)

target_correlation

## Step 4 - Visualize the target relationship

### Loan Status with Applicant Income

In [ ]:
sns.boxplot(
    x='loan_status',
    y='applicant_income',
    data=data
)

plt.title('Applicant Income by Loan Approval Status')
plt.xlabel('Loan Status')
plt.ylabel('Applicant Income')
plt.show()

### For: Loan Amount, Total Income and Loan Income Ratio

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

sns.boxplot(
    x='loan_status',
    y='loan_amount',
    data=data,
    ax=axes[0]
)

sns.boxplot(
    x='loan_status',
    y='total_income',
    data=data,
    ax=axes[1]
)

sns.boxplot(
    x='loan_status',
    y='loan_income_ratio',
    data=data,
    ax=axes[2]
)

axes[0].set_title('Loan Amount by Loan Status')
axes[1].set_title('Total Income by Loan Status')
axes[2].set_title('Loan-Income Ratio by Loan Status')

plt.tight_layout()
plt.show()

## Step 5 - Categorical/binary feature relationships

### Credit History

In [ ]:
credit_history_approval = pd.crosstab(
    data['credit_history'],
    data['loan_status'],
    normalize='index'
)

credit_history_approval

In [ ]:
credit_history_approval.plot(
    kind='bar',
    figsize=(8, 5)
)

plt.title('Loan Approval Rate by Credit History')
plt.xlabel('Credit History')
plt.ylabel('Proportion')
plt.legend(title='Loan Status')
plt.show()

## Step 6 - Feature Importance

**We'll use a Random Forest Classifier**

### Train/test split

In [ ]:
from sklearn.model_selection import train_test_split

X = data.drop(columns=['loan_status'])
y = data['loan_status']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

### Build the Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

rf_model.fit(X_train, y_train)

We are trying to build a tool for understanding feature importance

## Extract Feature Importance

In [ ]:
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by='importance',
    ascending=False
)

feature_importance

In [ ]:
plt.figure(figsize=(10, 7))

sns.barplot(
    data=feature_importance,
    x='importance',
    y='feature'
)

plt.title('Random Forest Feature Importance')
plt.xlabel('Importance')
plt.ylabel('Feature')

plt.show()

## ML-Ready dataset

In [ ]:
model_data = data.drop(
    columns=['applicant_income']
)

ml_ready_output_path = PROCESSED_DATA_DIR / 'loan_prediction_ml_ready.csv'
model_data.to_csv(ml_ready_output_path, index=False)

## Preprocessing Verification

In [ ]:
print('CLEANED DATASET')
print('=' * 40)
print(f'Rows: {cleaned_data.shape[0]}')
print(f'Columns: {cleaned_data.shape[1]}')
print(f'Missing values: {cleaned_data.isnull().sum().sum()}')
print(f'Duplicate rows: {cleaned_data.duplicated().sum()}')

print('\nML-READY DATASET')
print('=' * 40)
print(f'Rows: {model_data.shape[0]}')
print(f'Columns: {model_data.shape[1]}')
print(f'Missing values: {model_data.isnull().sum().sum()}')
print(f'Duplicate rows: {model_data.duplicated().sum()}')
print(f'Fully numeric: {all(pd.api.types.is_numeric_dtype(dtype) for dtype in model_data.dtypes)}')